In [1]:
import numpy as np
import numpy.typing as npt
import pandas as pd
import matplotlib.pyplot as plt
from structures import *
from pathlib import Path

In [2]:
I = parse_instance(Path("./instances/1000/competition/instance61_nreq1000_nveh20_gamma879.txt"))

In [3]:
def calc_my_metric(I:Instance, a:float)->npt.NDArray:
    n = I.n
    costs = np.zeros(n)
    max_dist = np.max(I.dist)    # over all requests
    max_dem  = np.max(I.demands)


    for req in range(n):
        p = 1 + req
        d = 1 + n + req
        solo_dist = (I.dist[0, p] + I.dist[p, d] + I.dist[d, 0])
        costs[req] = a * (solo_dist / max_dist) + (1-a) * (I.demands[req] / max_dem)
    
    return costs


def construction(I: Instance, a: float, sigma_factor: float = 0.1, is_random:bool=False) -> Solution:
    costs = calc_my_metric(I, a)


    if is_random:
        sigma = sigma_factor * (costs.std() if costs.std() > 0 else 1.0)
        noisy_costs = costs + np.random.normal(0, sigma, size=I.n)
    else:
        noisy_costs = costs


    perm = np.argsort(noisy_costs)
    important = perm[:I.gamma]

    per_track_requests = [important[i::I.nK] for i in range(I.nK)]

    routes = []
    for track in range(I.nK):
        route = []
        cargo = 0
        active = []

        for req in per_track_requests[track]:
            pickup = 1 + req
            dem = I.demands[req]

            # capacity check: drop heaviest if needed
            if cargo + dem > I.C:
                if not active:
                    continue
                heaviest = max(active, key=lambda r: I.demands[r])
                active.remove(heaviest)
                route.append(1 + I.n + heaviest)
                cargo -= I.demands[heaviest]

            # pick this request
            route.append(pickup)
            active.append(req)
            cargo += dem

        # drop remaining active requests
        active.sort(key=lambda r: I.demands[r], reverse=True)
        for r in active:
            route.append(1 + I.n + r)

        routes.append(route)

    return Solution(routes=routes)


def beam_search(I: Instance, a: float, beam_width: int = 5) -> Solution:
    costs = calc_my_metric(I, a)

    perm = np.argsort(costs)
    gamma = min(I.gamma, I.n)
    important = perm[:gamma]
    per_track_requests = [important[i::I.nK] for i in range(I.nK)]

    n = I.n
    routes = []

    for track in range(I.nK):

        # Each element: (score, route, cargo, active_set, remaining_requests)
        remaining = list(per_track_requests[track])
        partial_routes = [(0, [], 0, frozenset(), tuple(remaining))]

        for _ in range(len(remaining)):

            new_beam = []

            for score, route, cargo, active, rem in partial_routes:
                rem = list(rem)

                # ---- 1. Try picking any request still remaining ----
                for req in rem:
                    dem = I.demands[req]
                    if cargo + dem <= I.C:
                        p = 1 + req
                        new_route = route + [p]
                        new_cargo = cargo + dem
                        new_active = active | {req}
                        new_remaining = tuple(r for r in rem if r != req)
                        new_score = score + I.dist[p][0]  # crude g-score, improve later
                        new_beam.append((new_score, new_route, new_cargo, new_active, new_remaining))

                # ---- 2. Try dropping any active request ----
                for req in active:
                    d = 1 + n + req
                    new_route = route + [d]
                    new_cargo = cargo - I.demands[req]
                    new_active = frozenset(r for r in active if r != req)
                    new_remaining = tuple(rem)
                    new_score = score + I.dist[d][0]
                    new_beam.append((new_score, new_route, new_cargo, new_active, new_remaining))

            # prune beam
            new_beam.sort(key=lambda x: x[0])
            partial_routes = new_beam[:beam_width]

        # finalise each candidate by dropping all active requests
        best_score = float("inf")
        best_route = None

        for score, route, cargo, active, rem in partial_routes:
            final_route = list(route)
            # drop everything still active
            for req in sorted(active, key=lambda r: I.demands[r], reverse=True):
                final_route.append(1 + n + req)

            # evaluate full route length
            dist = route_distance(I, final_route)
            if dist < best_score:
                best_score = dist
                best_route = final_route

        routes.append(best_route)

    return Solution(routes=routes)




In [ ]:
def local_search(I:Instance):

SyntaxError: expected ':' (3271786888.py, line 1)

In [74]:
objective(I, sol)

128646.49800419911

22.19